In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import PowerTransformer

import re

In [ ]:
HEADERS = [
    "age",
    "workclass",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "hours-per-week",
    "native-country",
    "income"
]

In [ ]:
CATEGORICAL_HEADERS = [
    "workclass",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "native-country",
]

NUMERICAL_HEADERS = [
    "age",
    "education-num",
    "hours-per-week",
    "net-capital"
]

In [ ]:
region_map = {
    "united_states": "united_states",
    "outlying_us(guam_usvi_etc)": "united_states",
    "puerto_rico": "united_states",
    
    "mexico": "latin_america",
    "el_salvador": "latin_america",
    "cuba": "latin_america",
    "jamaica": "latin_america",
    "dominican_republic": "latin_america",
    "guatemala": "latin_america",
    "columbia": "latin_america",
    "haiti": "latin_america",
    "nicaragua": "latin_america",
    "peru": "latin_america",
    "ecuador": "latin_america",
    "trinadad&tobago": "latin_america",
    "honduras": "latin_america",
    
    "philippines": "asia",
    "india": "asia",
    "china": "asia",
    "japan": "asia",
    "vietnam": "asia",
    "taiwan": "asia",
    "thailand": "asia",
    "hong": "asia",
    "cambodia": "asia",
    "laos": "asia",
    
    "germany": "europe",
    "england": "europe",
    "italy": "europe",
    "poland": "europe",
    "portugal": "europe",
    "greece": "europe",
    "ireland": "europe",
    "france": "europe",
    "yugoslavia": "europe",
    "scotland": "europe",
    "hungary": "europe",
    "holand_netherlands": "europe",
    
    "canada": "canada",
    
    "south": "other",
    "iran": "other",
}

In [ ]:
from ucimlrepo import fetch_ucirepo 

adult = fetch_ucirepo(id=2) 

df = adult.data.features.copy()
df["income"] = adult.data.targets.copy()

In [ ]:
df = df.replace(r"^\s+|\s+$", "", regex=True)
df = df.replace("?", pd.NA)

In [ ]:
df["net-capital"] = df["capital-gain"] - df["capital-loss"]

In [ ]:
df.drop(["capital-gain", "capital-loss", "fnlwgt", "education"], inplace=True, axis=1)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
for i in range(len(HEADERS)):
    print(HEADERS[i])
    print(df[HEADERS[i]].unique())
    print("#"*50)

In [ ]:
for i in range(len(HEADERS)):
    nan = df[HEADERS[i]].isna().sum()
    if nan > 0:
        print(HEADERS[i])
        print(nan)
        print("#"*50)

In [ ]:
df = df.dropna()

In [ ]:
df.shape

In [ ]:
def text_normalizer(text):
    text = text.strip()
    text = text.replace("-", "_")
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text

In [ ]:
df[CATEGORICAL_HEADERS] = df[CATEGORICAL_HEADERS].map(text_normalizer)

In [ ]:
sex_map = {
    "Male" : 0,
    "Female" : 1
}

earning_map = {
    "<=50K" : 0,
    "<=50K." : 0,
    ">50K" : 1,
    ">50K." : 1
}

df["sex"] = df["sex"].map(sex_map)
df["income"] = df["income"].map(earning_map)

In [ ]:
for i in range(len(NUMERICAL_HEADERS)):
    print("skew \t",df[NUMERICAL_HEADERS[i]].skew())
    print("kurt \t",df[NUMERICAL_HEADERS[i]].kurt())
    print(df[NUMERICAL_HEADERS[i]].describe())
    print("#"*50)

In [ ]:
df["native-country"] = df["native-country"].map(region_map)

In [ ]:
df["occupation"] = df["occupation"].replace({
    "armed_forces" : "other_service",
})

df["marital-status"] = df["marital-status"].replace({
    "married_civ_spouse" : "married_spouse",
    "married_af_spouse" : "married_spouse",
})

In [ ]:
for i in range(len(CATEGORICAL_HEADERS)):
    print(df[CATEGORICAL_HEADERS[i]].value_counts())
    print("#"*50)

In [ ]:
df.head()

In [ ]:
transformer = PowerTransformer(method='yeo-johnson')

df["net-capital"] = transformer.fit_transform(df[["net-capital"]])

In [ ]:
for i in range(len(NUMERICAL_HEADERS)):
    print("skew \t",df[NUMERICAL_HEADERS[i]].skew())
    print("kurt \t",df[NUMERICAL_HEADERS[i]].kurt())
    print(df[NUMERICAL_HEADERS[i]].describe())
    print("#"*50)

In [ ]:
df = pd.get_dummies(df, columns=CATEGORICAL_HEADERS, dtype=np.int64)

In [ ]:
df.head()

In [ ]:
df.to_csv("preprocessed_data.csv", index=False)